# Turkish Earthquake Discourse Frame Detection

End-to-end run: install deps → mount Drive → TF-IDF baseline → BERTurk fine-tune → score full corpus.

**Before running:** set Runtime → Change runtime type → **GPU (T4 is fine, A100 if available)**.

**Drive layout expected:**
```
MyDrive/deprem/
  annotation/final_300.csv
  corpus/corpus_final.csv
MyDrive/nlp_pipeline/         
```

 dependencies





In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.18" "accelerate>=0.27" scikit-learn pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/nlp_pipeline')

# Sanity check: confirm files are visible
import os
for p in [
    '/content/drive/MyDrive/nlp_pipeline/config.py',
    '/content/drive/MyDrive/deprem/annotation/final_300.csv',
    '/content/drive/MyDrive/deprem/corpus/corpus_final.csv',
]:
    print(('OK   ' if os.path.exists(p) else 'MISS '), p)

Mounted at /content/drive
OK    /content/drive/MyDrive/nlp_pipeline/config.py
OK    /content/drive/MyDrive/deprem/annotation/final_300.csv
OK    /content/drive/MyDrive/deprem/corpus/corpus_final.csv


available GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


## TF-IDF + LinearSVR baseline





In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_tfidf_svr.py

/content/drive/MyDrive/nlp_pipeline
Loading training data: /content/drive/MyDrive/deprem/annotation/final_300.csv
  300 docs, frames: ['technical', 'political', 'development', 'sustainability']

=== 5-fold CV (TF-IDF + LinearSVR) ===
  [technical] CV MAE=0.534 ± 0.073 | QWK=0.670 ± 0.051
  [political] CV MAE=0.574 ± 0.056 | QWK=0.689 ± 0.092
  [development] CV MAE=0.638 ± 0.032 | QWK=0.429 ± 0.101
  [sustainability] CV MAE=0.210 ± 0.050 | QWK=0.127 ± 0.120

=== Summary ===
                  mae    qwk
technical       0.534  0.670
political       0.574  0.689
development     0.638  0.429
sustainability  0.210  0.127

=== Refitting on full training set for feature inspection ===

  Top-20 positive features for [technical]:
    +1.470  kentsel
    +1.297  bin
    +1.097  dönüşüm
    +0.997  köy
    +0.913  kentsel dönüşüm
    +0.748  zemin
    +0.740  risk
    +0.712  çelik
    +0.701  konut
    +0.694  su
    +0.637  bağımsız
    +0.634  şekilde
    +0.611  inşa
    +0.609  hasarlı
    +

## BERTurk fine-tune



In [ ]:
import re

path = "/content/drive/MyDrive/nlp_pipeline/pipeline_berturk.py"

with open(path, "r", encoding="utf-8") as f:
    code = f.read()

# "tokenizer=tokenizer" -> "processing_class=tokenizer"
code = code.replace("tokenizer=tokenizer,", "processing_class=tokenizer,")

with open(path, "w", encoding="utf-8") as f:
    f.write(code)

print("pipeline_berturk.py updated")

pipeline_berturk.py updated


In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_berturk.py

/content/drive/MyDrive/nlp_pipeline
Device: cuda
Train: 240 | Test: 60
Loading tokenizer + model: dbmdz/bert-base-turkish-cased
Loading weights: 100% 199/199 [00:00<00:00, 1128.19it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when 

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_tfidf_v2.py

/content/drive/MyDrive/nlp_pipeline
Traceback (most recent call last):
  File "/content/drive/MyDrive/nlp_pipeline/pipeline_tfidf_v2.py", line 12, in <module>
    import pandas as pd
  File "/usr/local/lib/python3.12/dist-packages/pandas/__init__.py", line 142, in <module>
    from pandas.io.api import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/api.py", line 6, in <module>
    from pandas.io.excel import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/excel/__init__.py", line 1, in <module>
    from pandas.io.excel._base import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/excel/_base.py", line 1435, in <module>
    class ExcelFile:
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/excel/_base.py", line 1495, in ExcelFile
    from pandas.io.excel._openpyxl import OpenpyxlReader
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<froz

In [ ]:
path = '/content/drive/MyDrive/nlp_pipeline/pipeline_tfidf_v2.py'
with open(path, 'r', encoding='utf-8') as f:
    code = f.read()

# GBR satırını yorum satırı yap
code = code.replace(
    '("tfidf_v2_gbr",             "gbr",       False, "GradientBoostingRegressor 200 trees"),',
    '# ("tfidf_v2_gbr",             "gbr",       False, "GradientBoostingRegressor 200 trees"),  # too slow'
)

with open(path, 'w', encoding='utf-8') as f:
    f.write(code)
print('GBR removed from config')

GBR removed from config


In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_berturk_cv.py --name berturk_cv_baseline --epochs 4 --lr 2e-5

/content/drive/MyDrive/nlp_pipeline
Device: cuda
Experiment: berturk_cv_baseline
Model: dbmdz/bert-base-turkish-cased | epochs=4 | lr=2e-05 | folds=5
Training data: 300 docs
Loading tokenizer: dbmdz/bert-base-turkish-cased

--- Fold 1/5 ---
  train=240 test=60
Loading weights: 100% 199/199 [00:00<00:00, 1509.43it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias            

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_berturk_cv.py --name berturk_cv_ep10_lr1e5 --epochs 10 --lr 1e-5

/content/drive/MyDrive/nlp_pipeline
Device: cuda
Experiment: berturk_cv_ep10_lr1e5
Model: dbmdz/bert-base-turkish-cased | epochs=10 | lr=1e-05 | folds=5
Training data: 300 docs
Loading tokenizer: dbmdz/bert-base-turkish-cased

--- Fold 1/5 ---
  train=240 test=60
Loading weights: 100% 199/199 [00:00<00:00, 1319.03it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight       

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_berturk_cv.py --name berturk_cv_distilbert_ep10_lr3e5 --epochs 10 --lr 3e-5 --model dbmdz/distilbert-base-turkish-cased

/content/drive/MyDrive/nlp_pipeline
Device: cuda
Experiment: berturk_cv_distilbert_ep10_lr3e5
Model: dbmdz/distilbert-base-turkish-cased | epochs=10 | lr=3e-05 | folds=5
Training data: 300 docs
Loading tokenizer: dbmdz/distilbert-base-turkish-cased
config.json: 100% 410/410 [00:00<00:00, 2.06MB/s]
tokenizer_config.json: 100% 60.0/60.0 [00:00<00:00, 149kB/s]
vocab.txt: 251kB [00:00, 22.1MB/s]

--- Fold 1/5 ---
  train=240 test=60
model.safetensors: 100% 273M/273M [00:06<00:00, 45.3MB/s]
Loading weights: 100% 100/100 [00:00<00:00, 1332.58it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: dbmdz/distilbert-base-turkish-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_class

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_berturk_cv_weighted.py --name distilbert_cv_weighted_ep10_lr3e5 --epochs 10 --lr 3e-5 --model dbmdz/distilbert-base-turkish-cased

/content/drive/MyDrive/nlp_pipeline
Device: cuda
Experiment: distilbert_cv_weighted_ep10_lr3e5
Model: dbmdz/distilbert-base-turkish-cased | epochs=10 | lr=3e-05 | folds=5
Training data: 300 docs
Loading tokenizer: dbmdz/distilbert-base-turkish-cased

--- Fold 1/5 ---
  train=240 test=60
  computing frame weights on train fold...
  [technical] reweighting: counts={0: 117, 1: 64, 2: 39, 3: 20} -> weight range [0.51, 3.00]
  [sustainability] reweighting: counts={0: 218, 1: 13, 2: 6, 3: 3} -> weight range [0.28, 20.00]
Map: 100% 240/240 [00:00<00:00, 440.68 examples/s]
Map: 100% 60/60 [00:00<00:00, 462.49 examples/s]
Loading weights: 100% 100/100 [00:00<00:00, 870.82it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: dbmdz/distilbert-base-turkish-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/deprem/outputs/experiments.csv', encoding='utf-8-sig')
df[df['frame'] == 'MEAN'][['experiment', 'model_family', 'qwk_mean', 'mae_mean']].sort_values('qwk_mean', ascending=False)

,experiment,model_family,qwk_mean,mae_mean
9,tfidf_v2_linearsvr_w,tfidf,0.6029,0.4853
4,tfidf_v2_linearsvr,tfidf,0.5919,0.4817
14,tfidf_v2_ridge,tfidf,0.5504,0.4968
19,tfidf_v2_ridge_w,tfidf,0.5463,0.5248
39,distilbert_cv_weighted_ep10_lr3e5,berturk,0.5334,0.1927
34,berturk_cv_distilbert_ep10_lr3e5,berturk,0.5190,0.1698
29,berturk_cv_ep10_lr1e5,berturk,0.3837,0.2032
24,berturk_cv_baseline,berturk,0.3543,0.1997


## models


In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python train_final_models.py

/content/drive/MyDrive/nlp_pipeline
Loaded 300 labeled docs

=== Training final TF-IDF v2 (LinearSVR + class weights) on full 300 docs ===
  feature matrix: (300, 31878)
  [technical] trained (weight range [0.52, 2.68])
  [political] trained (weight range [1.00, 1.00])
  [development] trained (weight range [1.00, 1.00])
  [sustainability] trained (weight range [0.27, 15.00])
  saved -> /content/drive/MyDrive/deprem/outputs/final_tfidf/tfidf_bundle.joblib

=== Training final DistilBERTurk (weighted MSE) on full 300 docs ===
  device: cuda
  [technical] reweighting: counts={0: 144, 1: 79, 2: 49, 3: 28} -> weight range [0.52, 2.68]
  [sustainability] reweighting: counts={0: 273, 1: 15, 2: 7, 3: 5} -> weight range [0.27, 15.00]
  weight matrix shape: (300, 4)
Map: 100% 300/300 [00:00<00:00, 468.07 examples/s]
Loading weights: 100% 100/100 [00:00<00:00, 2035.72it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from:

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python predict_corpus_both.py

/content/drive/MyDrive/nlp_pipeline
Loading corpus: /content/drive/MyDrive/deprem/corpus/corpus_final.csv
  2139 docs

=== Scoring with TF-IDF v2 weighted ===
Loading TF-IDF bundle: /content/drive/MyDrive/deprem/outputs/final_tfidf/tfidf_bundle.joblib

=== Scoring with DistilBERTurk weighted ===
Loading DistilBERTurk: /content/drive/MyDrive/deprem/outputs/final_distilbert
Loading weights: 100% 104/104 [00:00<00:00, 1744.04it/s, Materializing param=pre_classifier.weight]
  distilbert: scored 16/2139
  distilbert: scored 336/2139
  distilbert: scored 656/2139
  distilbert: scored 976/2139
  distilbert: scored 1296/2139
  distilbert: scored 1616/2139
  distilbert: scored 1936/2139

Writing: /content/drive/MyDrive/deprem/outputs/analytic_with_scores.csv
  2139 rows, 19 columns

=== Summary statistics ===

TF-IDF (0-3 scale):
       tfidf_score_technical  tfidf_score_political  tfidf_score_development  tfidf_score_sustainability
count               2139.000               2139.000           

In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/deprem/outputs/analytic_with_scores.csv', encoding='utf-8-sig')
print(f"Shape: {df.shape}")
print(f"\nİlk 5 satır:")
df[['doc_id', 'province',
    'tfidf_score_technical', 'distilbert_score_technical_0to3',
    'tfidf_score_sustainability', 'distilbert_score_sustainability_0to3']].head()

Shape: (2139, 19)

İlk 5 satır:


,doc_id,province,tfidf_score_technical,distilbert_score_technical_0to3,tfidf_score_sustainability,distilbert_score_sustainability_0to3
0,afad_045,national,1.524654,2.031519,0.337546,1.139248
1,adiyaman_val_0017,adiyaman,0.186446,0.000000,0.000000,0.000000
2,adiyaman_val_0019,adiyaman,0.030561,0.547564,0.000494,0.039854
3,adiyaman_val_0021,adiyaman,0.361445,0.473386,0.000000,0.400455
4,adiyaman_val_0023,adiyaman,0.260027,0.841631,0.147253,0.219917


## 7. Peek at the scored corpus

In [ ]:
import pandas as pd
scored = pd.read_csv('/content/drive/MyDrive/deprem/outputs/analytic_with_scores.csv', encoding='utf-8-sig')
print(scored.shape)
scored[['doc_id','province','score_technical','score_political','score_development','score_sustainability']].head(10)